# Step 5: Automate deployment with a SageMaker Project

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 4.3.2</strong> and <strong>Python 3.12</strong>.
</div>

Notebook 04 deployed one approved Model Package version explicitly so you could see every hosting object and connect data capture to monitoring. Here you turn that deployment into a reusable **continuous-delivery toolchain**: pass a **Model Package Group ARN** once, and every subsequent approval in that group can start the deployment pipeline.

There is already an AWS-provided deployment-only SageMaker Project template in this account:

> **`[Not supported] MLOps template for model deployment`**

It creates exactly the architecture needed here:

```text
Model Package approval ──▶ EventBridge ──▶ CodePipeline
                                                │
CodeCommit (deployment IaC) ──▶ EventBridge ────┤
                                                ▼
                                         Build templates
                                                ▼
                                         Deploy staging
                                                ▼
                                          Test staging
                                                ▼
                                         Manual approval
                                                ▼
                                        Deploy production
```

The Project provisions a **CodeCommit** repository containing `build.py`, `buildspec.yml`, staging/production configuration, an endpoint CloudFormation template, and a staging test. CodeBuild resolves the newest `Approved` version in the configured group and generates the deployment artifacts. CodePipeline deploys staging, tests it, waits for a human, and then deploys production.

## An important support caveat

AWS removed the CodeCommit-based SageMaker Project templates for new projects on **2024-10-28** and labels this product **not supported**. It remains present and launchable in this workshop account, whose CodeCommit access predates the restriction. Reusing it here is useful for understanding the canonical SageMaker Projects deployment pattern, but do not make an unsupported AWS-owned product the long-term foundation of a new production platform. For a supported implementation, copy the pattern into an organization-owned template and use your current source-control standard.

The exact CloudFormation artifact from this account is saved as [`cfn/endpoint_deployment_toolchain_template.yml`](cfn/endpoint_deployment_toolchain_template.yml), with provenance and limitations in [`cfn/README.md`](cfn/README.md). The notebook still launches through SageMaker Projects and Service Catalog—the normal path—so SageMaker supplies the generated project ID and associates all resources with the Project.

<div class="alert alert-warning">
<b>Cost and resource warning.</b> Creating the Project creates CodeCommit, CodeBuild, CodePipeline, EventBridge, CloudFormation and S3 resources. Its first pipeline execution deploys a <b>staging real-time endpoint</b>; approving the manual gate deploys a second, production endpoint. Real-time endpoints bill while they are <code>InService</code>. All mutating cells are disabled by default and require an explicit flag.
</div>

<div style="border: 4px solid #d32f2f; border-radius: 10px; padding: 20px; background-color: #ffebee; margin: 15px 0;">

### ⛔ IMPORTANT: Python 3.12 Required

All notebooks in this workshop must run on Python 3.12. If the next cell fails, run `./setup-py312.sh`, refresh JupyterLab, and select the **Python 3.12** kernel.

</div>

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), (
    f"Expected Python 3.12, got {sys.version_info.major}.{sys.version_info.minor}"
)

## Setup

Restore the AWS and Model Registry coordinates produced by the earlier notebooks.

In [ ]:
%store -r

try:
    initialized
except NameError:
    raise RuntimeError("Run 00-start-here.ipynb before this notebook.") from None

if "registered_model_name" not in dir():
    raise RuntimeError("Run notebooks 01 and 03 first; registered_model_name is not in %store.")

In [ ]:
import hashlib
import json
import os
import re
import time
from collections import Counter
from pathlib import Path

import boto3
import yaml
from botocore.exceptions import ClientError
from IPython.display import HTML, display

from governance.approval_gate import resolve_model_package_group

sm_client = boto3.client("sagemaker", region_name=region)
sc_client = boto3.client("servicecatalog", region_name=region)
codepipeline_client = boto3.client("codepipeline", region_name=region)
codecommit_client = boto3.client("codecommit", region_name=region)
cfn_client = boto3.client("cloudformation", region_name=region)
s3_client = boto3.client("s3", region_name=region)
account_id = boto3.client("sts").get_caller_identity()["Account"]

print(f"Region:            {region}")
print(f"Account:           {account_id}")
print(f"Registered model:  {registered_model_name}")

---
## 1. The input: a Model Package Group ARN

The Service Catalog template itself asks for `SourceModelPackageGroupName`, not an ARN. An ARN is a safer interface for automation because it is unambiguous about partition, Region, account and resource type. This notebook accepts the ARN, validates that it belongs to this account and Region, and passes only its final path component to the AWS template.

Leave the value empty to use the group created by this workshop, or paste the ARN of another group in the same account and Region.

In [ ]:
MODEL_PACKAGE_GROUP_ARN = ""  # e.g. arn:aws:sagemaker:us-west-2:123456789012:model-package-group/my-group

if not MODEL_PACKAGE_GROUP_ARN:
    workshop_group_name = resolve_model_package_group(
        sm_client, registered_model_name, prefix="from-idea-to-prod-model-"
    )
    MODEL_PACKAGE_GROUP_ARN = sm_client.describe_model_package_group(
        ModelPackageGroupName=workshop_group_name
    )["ModelPackageGroupArn"]
    print("No override supplied; using the workshop group.")

print(f"Model Package Group ARN: {MODEL_PACKAGE_GROUP_ARN}")

In [ ]:
match = re.fullmatch(
    r"arn:(?P<partition>aws(?:-[a-z]+)*):sagemaker:(?P<region>[^:]+):"
    r"(?P<account>\d{12}):model-package-group/(?P<name>[A-Za-z0-9](?:-*[A-Za-z0-9]){0,62})",
    MODEL_PACKAGE_GROUP_ARN,
)
if not match:
    raise ValueError("Expected a SageMaker Model Package Group ARN, not a version ARN or a group name.")
if match["region"] != region or match["account"] != account_id:
    raise ValueError(
        f"The AWS template deploys in {account_id}/{region}, but the ARN belongs to "
        f"{match['account']}/{match['region']}."
    )

model_package_group_name = match["name"]
group = sm_client.describe_model_package_group(ModelPackageGroupName=model_package_group_name)
assert group["ModelPackageGroupArn"] == MODEL_PACKAGE_GROUP_ARN

print(f"Validated group name passed to Service Catalog: {model_package_group_name}")
print(f"Group status: {group['ModelPackageGroupStatus']}")

The product's `build.py` deploys the newest **`Approved`** package in the group. It does not understand the workshop's lifecycle stage, so check both attributes before creating the automation. This is a preflight guard in the notebook; the unmodified AWS seed code still filters on approval alone.

In [ ]:
approved = sm_client.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    ModelApprovalStatus="Approved",
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=100,
)["ModelPackageSummaryList"]
if not approved:
    raise RuntimeError(f"No Approved Model Package exists in {model_package_group_name}.")

candidate_arn = approved[0]["ModelPackageArn"]
candidate = sm_client.describe_model_package(ModelPackageName=candidate_arn)
lifecycle = candidate.get("ModelLifeCycle") or {}
if lifecycle.get("Stage") != "production" or lifecycle.get("StageStatus") != "active":
    raise RuntimeError(
        f"Latest Approved package is not production/active: {candidate_arn} -> {lifecycle}. "
        "Finish the governance gate in notebook 03."
    )
if not candidate.get("InferenceSpecification"):
    raise RuntimeError(f"{candidate_arn} has no inference specification and cannot be deployed.")

print("Deployment candidate selected by the AWS seed code:")
print(f"  Package:   {candidate_arn}")
print(f"  Approval:  {candidate['ModelApprovalStatus']}")
print(f"  Lifecycle: {lifecycle['Stage']} / {lifecycle['StageStatus']}")
print(f"  Image:     {candidate['InferenceSpecification']['Containers'][0]['Image']}")

---
## 2. Inspect the exact AWS toolchain template saved locally

The file below is not an approximation: it was retrieved with `cloudformation:GetTemplate` from a temporary Project created from this product and artifact. Assert its provenance checksum and inspect its resource graph before trusting it.

In [ ]:
template_path = Path("cfn/endpoint_deployment_toolchain_template.yml")
template_bytes = template_path.read_bytes()
template_sha256 = hashlib.sha256(template_bytes).hexdigest()
expected_sha256 = "1a6349150e4b984641202718cbae55e1497f7185d1c14afeebf67fb825f5b960"
assert template_sha256 == expected_sha256, (
    f"Template changed: expected {expected_sha256}, got {template_sha256}. Review before launching."
)

toolchain = yaml.safe_load(template_bytes)
resource_types = Counter(resource["Type"] for resource in toolchain["Resources"].values())
expected_parameters = {
    "SageMakerProjectName", "SageMakerProjectId", "SourceModelPackageGroupName"
}
assert set(toolchain["Parameters"]) == expected_parameters
assert resource_types["AWS::CodeCommit::Repository"] == 1
assert resource_types["AWS::CodePipeline::Pipeline"] == 1

print(f"Template: {template_path} ({len(template_bytes):,} bytes)")
print(f"SHA-256:  {template_sha256}")
print(f"Parameters: {sorted(toolchain['Parameters'])}")
print("Resources:")
for resource_type, count in sorted(resource_types.items()):
    print(f"  {count} x {resource_type}")

In [ ]:
repository = toolchain["Resources"]["ModelDeployCodeCommitRepository"]
pipeline = toolchain["Resources"]["ModelDeployPipeline"]
stages = [stage["Name"] for stage in pipeline["Properties"]["Stages"]]

print("CodeCommit seed:")
print(f"  bucket: {repository['Properties']['Code']['S3']['Bucket']}")
print(f"  key:    {repository['Properties']['Code']['S3']['Key']}")
print(f"  branch: {repository['Properties']['Code']['BranchName']}")
print(f"CodePipeline stages: {' -> '.join(stages)}")

The template contains only the **toolchain**. At Project creation it seeds CodeCommit from `mpg-deployment-config-v1.1.zip`. That repository contains the endpoint template and stage configuration used by the pipeline. The generated endpoint template creates a SageMaker `Model` from the selected Model Package, an `EndpointConfig` with input/output data capture, and an `Endpoint` named `<project>-staging` or `<project>-prod`.

Two details matter:

- The artifact bucket has `DeletionPolicy: Retain`; deleting the Project does not remove it automatically.
- The model event rule matches approval status `anything-but PendingManualApproval`, not only `Approved`. A rejection can therefore rerun the pipeline, but `build.py` still resolves the newest package whose status is actually `Approved`.

---
## 3. Discover the existing Service Catalog product

Do not hard-code the regional product IDs into the launch call. Find the exact product by name, select its newest active artifact, find an available launch path, and verify its parameter contract. The IDs are printed for auditability and should match the snapshot metadata in `cfn/README.md` in this account.

In [ ]:
PRODUCT_NAME = "[Not supported] MLOps template for model deployment"

product_matches = []
page_token = None
while True:
    request = {"Filters": {"FullTextSearch": [PRODUCT_NAME]}, "PageSize": 100}
    if page_token:
        request["PageToken"] = page_token
    page = sc_client.search_products(**request)
    product_matches.extend(
        product for product in page["ProductViewSummaries"] if product["Name"] == PRODUCT_NAME
    )
    page_token = page.get("NextPageToken")
    if not page_token:
        break
if len(product_matches) != 1:
    raise LookupError(
        f"Expected one visible Service Catalog product named {PRODUCT_NAME!r}, found {len(product_matches)}. "
        "The unsupported CodeCommit template may have been removed from this account."
    )

product = product_matches[0]
product_id = product["ProductId"]
artifacts = sc_client.list_provisioning_artifacts(ProductId=product_id)["ProvisioningArtifactDetails"]
active_artifacts = [artifact for artifact in artifacts if artifact.get("Active", True)]
artifact = max(active_artifacts, key=lambda item: item["CreatedTime"])
artifact_id = artifact["Id"]

paths = sc_client.list_launch_paths(ProductId=product_id, PageSize=20)["LaunchPathSummaries"]
if not paths:
    raise PermissionError(f"Product {product_id} is visible but has no launch path for this role.")
path_id = paths[0]["Id"]

print(f"Product:  {product['Name']}")
print(f"ID:       {product_id}")
print(f"Artifact: {artifact['Name']} / {artifact_id} ({artifact.get('Description', '')})")
print(f"Path:     {paths[0]['Name']} / {path_id}")

In [ ]:
contract = sc_client.describe_provisioning_parameters(
    ProductId=product_id,
    ProvisioningArtifactId=artifact_id,
    PathId=path_id,
)
parameters = {parameter["ParameterKey"]: parameter for parameter in contract["ProvisioningArtifactParameters"]}
assert set(parameters) == expected_parameters

print("Service Catalog parameter contract:")
for key, parameter in parameters.items():
    print(f"  {key:<30}{parameter['Description']}")
print("\nLaunch constraint:")
for constraint in contract.get("ConstraintSummaries", []):
    print(f"  {constraint['Type']}: {constraint.get('Description', '')}")

`SageMakerProjectName` and `SageMakerProjectId` appear in the CloudFormation contract, but **do not pass them yourself** to `create_project()`: SageMaker generates the ID and injects both values. The only user provisioning parameter is the group name derived from the ARN.

---
## 4. Create the deployment Project

Use the normal SageMaker Projects API with `ServiceCatalogProvisioningDetails`. This creates the SageMaker Project association and asks Service Catalog to launch the product with its configured launch role.

Choose a stable name so rerunning the notebook finds and reuses the same Project rather than creating duplicates. Set `CREATE_PROJECT = True` only when you are ready for the staging endpoint and its hourly charge.

In [ ]:
PROJECT_NAME = "from-idea-prod-deploy"  # 1-32 chars, starts with a letter
CREATE_PROJECT = False                   # <-- explicit opt-in: creates CI/CD resources and a staging endpoint

if not re.fullmatch(r"[A-Za-z](?:-*[A-Za-z0-9]){0,31}", PROJECT_NAME):
    raise ValueError("PROJECT_NAME must satisfy ^[A-Za-z](-*[A-Za-z0-9])*, max 32 characters.")

existing_projects = {
    project["ProjectName"]: project
    for project in sm_client.list_projects(MaxResults=100)["ProjectSummaryList"]
}
project_created_here = False
project_details = None

if PROJECT_NAME in existing_projects:
    project_details = sm_client.describe_project(ProjectName=PROJECT_NAME)
    configured = {
        item["Key"]: item["Value"]
        for item in project_details["ServiceCatalogProvisioningDetails"].get("ProvisioningParameters", [])
    }
    if configured.get("SourceModelPackageGroupName") != model_package_group_name:
        raise RuntimeError(
            f"Existing project {PROJECT_NAME} targets {configured.get('SourceModelPackageGroupName')}, "
            f"not {model_package_group_name}. Choose another PROJECT_NAME."
        )
    print(f"Reusing existing project {PROJECT_NAME} ({project_details['ProjectStatus']}).")
elif not CREATE_PROJECT:
    print("PREVIEW ONLY — no resources created.")
    print(f"Project: {PROJECT_NAME}")
    print(f"Product/artifact/path: {product_id} / {artifact_id} / {path_id}")
    print(f"SourceModelPackageGroupName: {model_package_group_name}")
    print("Set CREATE_PROJECT = True and rerun this cell to provision it.")
else:
    response = sm_client.create_project(
        ProjectName=PROJECT_NAME,
        ProjectDescription=(
            "Deployment-only CI/CD for approved Model Packages: CodeCommit, CodeBuild, "
            "CodePipeline, staging test, manual production approval."
        ),
        ServiceCatalogProvisioningDetails={
            "ProductId": product_id,
            "ProvisioningArtifactId": artifact_id,
            "PathId": path_id,
            "ProvisioningParameters": [
                {"Key": "SourceModelPackageGroupName", "Value": model_package_group_name}
            ],
        },
        Tags=[
            {"Key": "workshop", "Value": "from-idea-to-production"},
            {"Key": "purpose", "Value": "model-deployment"},
        ],
    )
    project_created_here = True
    project_details = sm_client.describe_project(ProjectName=PROJECT_NAME)
    print(f"Creating {response['ProjectArn']}")

Project creation normally takes two to five minutes. SageMaker returns before Service Catalog is finished, so wait for `CreateCompleted`. When the preview flag is still false, the cell simply explains why it is skipped.

In [ ]:
if project_details is None:
    print("Skipped: enable CREATE_PROJECT first.")
else:
    terminal = {"CreateCompleted", "CreateFailed", "CreateCompletedWithErrors"}
    deadline = time.time() + 900
    while project_details["ProjectStatus"] not in terminal:
        if time.time() > deadline:
            raise TimeoutError(f"Project creation did not finish: {project_details['ProjectStatus']}")
        print(f"  {time.strftime('%H:%M:%S')} | {project_details['ProjectStatus']}")
        time.sleep(20)
        project_details = sm_client.describe_project(ProjectName=PROJECT_NAME)

    if project_details["ProjectStatus"] != "CreateCompleted":
        raise RuntimeError(
            f"Project creation failed: {project_details['ProjectStatus']} — "
            f"{project_details.get('ProjectStatusMessage', '')}"
        )
    project_id = project_details["ProjectId"]
    print(f"Project ready: {PROJECT_NAME} / {project_id}")

### Inspect the generated toolchain

Every resource name is deterministic from the Project name and generated ID. The CodeCommit repository is the part you own and customize; changing its `main` branch starts the same pipeline as a model approval event.

In [ ]:
if project_details is None or project_details.get("ProjectStatus") != "CreateCompleted":
    print("Skipped: no ready Project.")
else:
    project_id = project_details["ProjectId"]
    repository_name = f"sagemaker-{PROJECT_NAME}-{project_id}-modeldeploy"
    pipeline_name = repository_name
    artifact_bucket = f"sagemaker-project-{project_id}"
    staging_stack_name = f"sagemaker-{PROJECT_NAME}-{project_id}-deploy-staging"
    prod_stack_name = f"sagemaker-{PROJECT_NAME}-{project_id}-deploy-prod"
    staging_endpoint_name = f"{PROJECT_NAME}-staging"
    prod_endpoint_name = f"{PROJECT_NAME}-prod"

    repository_info = codecommit_client.get_repository(repositoryName=repository_name)["repositoryMetadata"]
    pipeline_info = codepipeline_client.get_pipeline(name=pipeline_name)["pipeline"]
    print(f"CodeCommit: {repository_info['cloneUrlHttp']}")
    print(f"CodePipeline: {pipeline_name} ({' -> '.join(stage['name'] for stage in pipeline_info['stages'])})")
    print(f"Artifacts: s3://{artifact_bucket}")
    print(f"Expected endpoints: {staging_endpoint_name}, {prod_endpoint_name}")

---
## 5. Start and observe a deployment

Project creation seeds the CodeCommit repository, and that commit normally starts the first pipeline execution. Model approvals after the Project exists start it through the model EventBridge rule. To force a deployment of an already-approved version—or to rerun after changing configuration—set `START_DEPLOYMENT = True`.

The pipeline first deploys staging and runs its test. The AWS seed test checks that the endpoint is `InService`; its `invoke_endpoint()` function is only scaffolding. Replace it with a real request and assertions for your model before relying on this gate in production.

In [ ]:
START_DEPLOYMENT = False  # starts CodeBuild and creates/updates the staging endpoint

pipeline_execution_id = None
if project_details is None or project_details.get("ProjectStatus") != "CreateCompleted":
    print("Skipped: no ready Project.")
else:
    executions = codepipeline_client.list_pipeline_executions(
        pipelineName=pipeline_name, maxResults=10
    )["pipelineExecutionSummaries"]
    if START_DEPLOYMENT:
        started = codepipeline_client.start_pipeline_execution(
            name=pipeline_name,
            clientRequestToken=f"notebook-{int(time.time())}",
        )
        pipeline_execution_id = started["pipelineExecutionId"]
        print(f"Started pipeline execution {pipeline_execution_id}")
    elif executions:
        pipeline_execution_id = executions[0]["pipelineExecutionId"]
        print(f"Observing latest existing execution {pipeline_execution_id} ({executions[0]['status']})")
    else:
        print("No execution exists. Set START_DEPLOYMENT = True to deploy the latest approved version.")

In [ ]:
def action_statuses(pipeline_state):
    return {
        (stage["stageName"], action["actionName"]): action.get("latestExecution", {}).get("status", "NotStarted")
        for stage in pipeline_state["stageStates"]
        for action in stage["actionStates"]
    }


def print_pipeline_state(pipeline_name):
    state = codepipeline_client.get_pipeline_state(name=pipeline_name)
    statuses = action_statuses(state)
    print("Pipeline actions:")
    for (stage, action), status in statuses.items():
        print(f"  {stage:<16}{action:<30}{status}")
    return state, statuses


if pipeline_execution_id:
    state, statuses = print_pipeline_state(pipeline_name)
else:
    print("Skipped: no pipeline execution to inspect.")

Run the next cell while an execution is active. It waits until staging and the test have succeeded and the manual approval action is waiting, or until the pipeline fails. A pipeline waiting at manual approval remains `InProgress` by design.

In [ ]:
WAIT_FOR_STAGING = False  # waits for a billable staging endpoint when a deployment is running

if not WAIT_FOR_STAGING or not pipeline_execution_id:
    print("Skipped: set WAIT_FOR_STAGING = True after starting a deployment.")
else:
    deadline = time.time() + 1800
    while True:
        execution = codepipeline_client.get_pipeline_execution(
            pipelineName=pipeline_name, pipelineExecutionId=pipeline_execution_id
        )["pipelineExecution"]
        state = codepipeline_client.get_pipeline_state(name=pipeline_name)
        statuses = action_statuses(state)
        test_status = statuses.get(("DeployStaging", "TestStaging"))
        approval_status = statuses.get(("DeployStaging", "ApproveDeployment"))
        print(
            f"  {time.strftime('%H:%M:%S')} | pipeline={execution['status']} "
            f"test={test_status} approval={approval_status}"
        )
        if execution["status"] in {"Failed", "Stopped", "Superseded", "Cancelled"}:
            raise RuntimeError(f"Pipeline ended before approval: {execution}")
        if test_status == "Succeeded" and approval_status == "InProgress":
            print(f"Staging passed; {staging_endpoint_name} is waiting for production approval.")
            break
        if execution["status"] == "Succeeded":
            print("Pipeline already completed.")
            break
        if time.time() > deadline:
            raise TimeoutError("Staging did not reach its manual approval gate within 30 minutes.")
        time.sleep(30)

### The production gate

Approval here is a **CodePipeline manual approval**, distinct from `ModelApprovalStatus=Approved` in the Model Registry:

- Model Registry approval allows this version into the deployment pipeline and triggers it.
- CodePipeline approval promotes the already-tested staging deployment to production.

In a real operating model, a deployment approver reviews the staging test and approves in the CodePipeline console. The API cell is included to make the flow reproducible, but is disabled by default.

In [ ]:
APPROVE_PRODUCTION = False  # creates/updates the billable production endpoint

if not APPROVE_PRODUCTION:
    print("Production approval disabled. Review staging in CodePipeline first.")
elif not pipeline_execution_id:
    raise RuntimeError("No pipeline execution is available to approve.")
else:
    state = codepipeline_client.get_pipeline_state(name=pipeline_name)
    approval_action = next(
        action
        for stage in state["stageStates"] if stage["stageName"] == "DeployStaging"
        for action in stage["actionStates"] if action["actionName"] == "ApproveDeployment"
    )
    latest = approval_action.get("latestExecution", {})
    if latest.get("status") != "InProgress" or "token" not in latest:
        raise RuntimeError(f"Approval action is not waiting: {latest}")
    codepipeline_client.put_approval_result(
        pipelineName=pipeline_name,
        stageName="DeployStaging",
        actionName="ApproveDeployment",
        token=latest["token"],
        result={
            "summary": f"Staging test reviewed from notebook 05 for {candidate_arn}",
            "status": "Approved",
        },
    )
    print(f"Approved production deployment for execution {pipeline_execution_id}")

In [ ]:
WAIT_FOR_PRODUCTION = False

if not WAIT_FOR_PRODUCTION or not pipeline_execution_id:
    print("Skipped: set WAIT_FOR_PRODUCTION = True after approving production.")
else:
    deadline = time.time() + 1800
    while True:
        execution = codepipeline_client.get_pipeline_execution(
            pipelineName=pipeline_name, pipelineExecutionId=pipeline_execution_id
        )["pipelineExecution"]
        print(f"  {time.strftime('%H:%M:%S')} | {execution['status']}")
        if execution["status"] in {"Succeeded", "Failed", "Stopped", "Superseded", "Cancelled"}:
            break
        if time.time() > deadline:
            raise TimeoutError("Production deployment did not finish within 30 minutes.")
        time.sleep(30)
    if execution["status"] != "Succeeded":
        raise RuntimeError(f"Production deployment failed: {execution}")
    endpoints = {
        endpoint["EndpointName"]: endpoint["EndpointStatus"]
        for endpoint in sm_client.list_endpoints(NameContains=PROJECT_NAME)["Endpoints"]
    }
    print(f"Staging:    {staging_endpoint_name} -> {endpoints.get(staging_endpoint_name, 'not found')}")
    print(f"Production: {prod_endpoint_name} -> {endpoints.get(prod_endpoint_name, 'not found')}")

### Continuous delivery from here on

The setup is now persistent:

- approving a new version in this Model Package Group emits a Model Package state-change event and starts the pipeline;
- committing deployment configuration or tests to CodeCommit `main` starts it too;
- `build.py` always resolves the newest `Approved` version at execution time;
- staging is automatic, production stays behind the manual CodePipeline gate.

To customize the generated repository in Studio, clone the URL printed above. The files to review first are `staging-config.json`, `prod-config.json`, `endpoint-config-template.yml`, and `test/test.py`.

---
## 6. Clean-up

The deployment endpoint stacks are **not children** of the SageMaker Project stack; CodePipeline creates them separately. Delete them before deleting the Project. The toolchain's S3 bucket is retained by design and is removed in a separate opt-in step. In this workshop account the Studio role can empty that bucket but lacks `s3:DeleteBucket`; the cell reports the remaining administrator action instead of hiding the failed deletion.

Never enable these flags for a Project you did not create for this workshop. The cells use the exact deterministic names printed above and refuse to do anything when no ready Project exists.

In [ ]:
DELETE_DEPLOYED_ENDPOINTS = False  # deletes staging/prod endpoint CloudFormation stacks
DELETE_PROJECT = False             # deletes CodeCommit/CodeBuild/CodePipeline/EventBridge toolchain
DELETE_RETAINED_BUCKET = False     # empties and deletes the retained artifact bucket

if project_details is None or project_details.get("ProjectStatus") != "CreateCompleted":
    print("No ready Project in this kernel; cleanup skipped.")
elif not any((DELETE_DEPLOYED_ENDPOINTS, DELETE_PROJECT, DELETE_RETAINED_BUCKET)):
    print("Cleanup preview:")
    print(f"  Endpoint stacks: {staging_stack_name}, {prod_stack_name}")
    print(f"  Project:         {PROJECT_NAME}")
    print(f"  Retained bucket: s3://{artifact_bucket}")
else:
    # Stop an execution before deleting resources underneath it.
    executions = codepipeline_client.list_pipeline_executions(
        pipelineName=pipeline_name, maxResults=10
    )["pipelineExecutionSummaries"]
    for execution in executions:
        if execution["status"] == "InProgress":
            codepipeline_client.stop_pipeline_execution(
                pipelineName=pipeline_name,
                pipelineExecutionId=execution["pipelineExecutionId"],
                abandon=True,
                reason="Notebook 05 cleanup",
            )
            print(f"Stopped pipeline execution {execution['pipelineExecutionId']}")

    if DELETE_DEPLOYED_ENDPOINTS:
        for stack_name in (staging_stack_name, prod_stack_name):
            try:
                cfn_client.describe_stacks(StackName=stack_name)
                cfn_client.delete_stack(StackName=stack_name)
                print(f"Deleting {stack_name}")
                cfn_client.get_waiter("stack_delete_complete").wait(StackName=stack_name)
            except ClientError as e:
                if "does not exist" not in str(e):
                    raise
                print(f"No stack: {stack_name}")

    if DELETE_PROJECT:
        sm_client.delete_project(ProjectName=PROJECT_NAME)
        print(f"Deleting Project {PROJECT_NAME}; Service Catalog removes the toolchain asynchronously.")
        deadline = time.time() + 900
        while True:
            try:
                status = sm_client.describe_project(ProjectName=PROJECT_NAME)["ProjectStatus"]
                print(f"  {time.strftime('%H:%M:%S')} | {status}")
            except ClientError as e:
                if e.response["Error"]["Code"] == "ValidationException":
                    print("Project deleted.")
                    break
                raise
            if status == "DeleteCompleted":
                print("Project deleted.")
                break
            if time.time() > deadline:
                raise TimeoutError(f"Project still exists with status {status}")
            time.sleep(20)

    if DELETE_RETAINED_BUCKET:
        if not DELETE_PROJECT:
            raise ValueError("Set DELETE_PROJECT = True together with DELETE_RETAINED_BUCKET.")
        # Handles both current and versioned objects. Some workshop roles can empty
        # this retained bucket but cannot call s3:DeleteBucket; that leaves an empty,
        # non-billable bucket and prints the administrator action required.
        bucket = boto3.resource("s3", region_name=region).Bucket(artifact_bucket)
        bucket.object_versions.delete()
        try:
            bucket.delete()
            print(f"Deleted retained bucket s3://{artifact_bucket}")
        except ClientError as e:
            if e.response["Error"]["Code"] != "AccessDenied":
                raise
            print(
                f"Emptied s3://{artifact_bucket}, but this role cannot delete buckets. "
                "Ask an administrator with s3:DeleteBucket to remove the empty bucket."
            )

## Summary

You reused the original SageMaker deployment-only Project product rather than rebuilding its control plane:

- a Model Package Group **ARN** is the notebook interface; its validated group name is the product input;
- the exact AWS CloudFormation artifact is versioned under `cfn/` and verified by checksum;
- SageMaker Projects launches the existing Service Catalog product, which provisions CodeCommit, CodeBuild, CodePipeline, EventBridge and S3;
- the latest approved Model Package is deployed to staging, tested, held behind a manual gate, then deployed to production;
- future model approvals and CodeCommit changes trigger the same pipeline.

This is genuine reuse, but of a retired CodeCommit product. The architecture is still sound; for a durable implementation, own the template and seed repository, strengthen `test/test.py`, restrict the event to your governance contract, and migrate source control to a supported option when required.

## Additional resources

- [Use SageMaker AI-provided project templates](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-projects-templates-sm.html)
- [MLOps Project Templates](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-projects-templates.html)
- [SageMaker Model Package state-change events](https://docs.aws.amazon.com/sagemaker/latest/dg/automating-sagemaker-with-eventbridge.html#eventbridge-model-package)
- [CodePipeline manual approval actions](https://docs.aws.amazon.com/codepipeline/latest/userguide/approvals.html)
- [Deploy a Model from the Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry-deploy.html)

# Shutdown kernel

In [ ]:
%%html
<p><b>Shutting down your kernel for this notebook.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
<script>
try { document.getElementsByClassName("sm-command-button")[0].click(); } catch(err) {}
</script>